# Lab 7.1 &mdash; Locating the Failing Step

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 50 min &nbsp;|&nbsp; **Day 3 &middot; Module 7 &mdash; Multi-Agent System Evaluation**

### What you'll do
- Read one run out of an audit trail that spans a whole conversation
- Walk a fixed ladder of eight checks, cheapest first, over six real runs
- Tell apart two runs that look equally wrong and failed at different steps
- Work out what each failure cost, and which one to escalate
- Break one step of your own deployment, locate it, and fix just that step

> **How this lab works.** You write real LangChain code &mdash; the agent under test, the callback
> handler that traces it, the typed verdict you grade. Fill every `BLANK`, then run the
> **Self-check** cell under each section. Those check the *objects you built* and the *recorded
> runs* shipped in the notebook, so they are deterministic and do not depend on the model.
> Cells marked **Run it for real** put your code in front of the sandbox model; that is the part
> worth watching, and it is never scored &mdash; scoring a live run would contradict Lab 7.1.

> **Six real runs.** They were captured from this app deployed on this cluster.
> Four of them are wrong, each at a different step, and every one of the six
> printed `QA PASS`. Your job is not to decide *whether* they are wrong &mdash;
> that is given &mdash; but to say *where*, from the evidence.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap, random, statistics
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-7-01")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens. Off is the default here because an eval lab makes a lot of calls.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

# ---- your own deployment, for Section 3 only -------------------------------
# Module 7's setup does not carry these, so read them here. Section 3 self-skips
# when they are absent, which is what an offline verification run sees.
import re as _re

APP_NS   = os.environ.get("APP_NAMESPACE", "")
APP_HOST = os.environ.get("APP_HOST", "")
print("namespace :", APP_NS or "(unset -- section 3 will self-skip)")

## Concept &mdash; three sources, and what each one cannot tell you

You have a bad answer. Somewhere in eight steps, something went wrong. Guessing is expensive,
so work from evidence, and know what each source is blind to.

| Source | Answers | Blind to |
|---|---|---|
| **the audit trail** (printed under every reply) | *which* steps ran, the category, the confidence, the tool calls, what QA did | how long anything took, and what the tool actually returned |
| **Prometheus / Grafana** (`frontdeskai_*`) | which step is slow or expensive, across many runs | this one request. A metric is a population, not a case |
| **the Tempo trace** | the real tree: what nested inside what, with timings and token counts | whether the *content* was right |

None of them says &ldquo;the answer was wrong&rdquo;. That judgement is yours; their job is to tell
you **where** it went wrong so you fix that step and not a different one.

### The six runs in this lab are real

They were captured from FrontDesk AI deployed on this cluster, by asking it six questions. The
audit trails below are what it actually printed. Four of the six are wrong in some way, each at a
different step, and &mdash; the thing worth sitting with &mdash; **all six passed the QA gate**.

In [ ]:
# ------------------------- six real runs, captured from the deployed app
# Nothing here is synthetic. These are the audit trails FrontDesk AI printed
# when it was asked these six questions on this cluster, with the timestamps
# removed. Each trail carries TWO turns, because that is how the app returns
# it -- the real arrays ran from 102 to 144 entries; they are cut to the last
# two turns here so the notebook stays readable.
RUNS = [
  {
    "id": 'run-1', "question": 'What is my current leave balance?',
    "category": 'hr', "confidence": 10,
    "escalated": False, "fallback_used": False,
    # the trail as returned: this turn PRECEDED by the one before it.
    "audit": [
      'Supervisor: tech (conf: 9)',
      'RAG: retrieved 4 chunks from 4 sources',
      'Few-shot: 0 example(s) retrieved',
      'Tech ReAct: 2 iteration(s), tools: list_my_tickets({})',
      'Tech worker: resolved',
      'Escalation check: False',
      'QA PASS',
      'Finalized for Rajesh Kumar',
      'Supervisor: hr (conf: 10)',
      'RAG: retrieved 4 chunks from 4 sources',
      'Few-shot: 0 example(s) retrieved',
      'Hr ReAct: 3 iteration(s), tools: get_leave_balance_from_hr_system({}), get_leave_balance({})',
      'Hr worker: resolved',
      'Escalation check: False',
      'QA PASS',
      'Finalized for Rajesh Kumar',
    ],
    "sources": ['Hr Handbook > Casual Leave (CL)', 'Hr Handbook > Earned/Privilege Leave (EL)', 'Hr Handbook > Sick Leave (SL)', 'Hr Handbook > Leave Policy'],
    "answer": '[HR] Your current leave balance is: 17 Casual Leave days, 8 Sick Leave days, 12 Earned Leave days, and 24 WFH days remaining. Sources: Hr Handbook > Casual Leave (CL), Hr Handbook > Earned/Privilege Leave (EL), Hr Handbook > Sick ',
  },
  {
    "id": 'run-2', "question": 'How do I claim for a laptop damaged on work travel?',
    "category": 'finance', "confidence": 8,
    "escalated": False, "fallback_used": False,
    # the trail as returned: this turn PRECEDED by the one before it.
    "audit": [
      'Supervisor: hr (conf: 10)',
      'RAG: retrieved 4 chunks from 4 sources',
      'Few-shot: 0 example(s) retrieved',
      'Hr ReAct: 3 iteration(s), tools: get_leave_balance_from_hr_system({}), get_leave_balance({})',
      'Hr worker: resolved',
      'Escalation check: False',
      'QA PASS',
      'Finalized for Rajesh Kumar',
      'Supervisor: finance (conf: 8)',
      'RAG: retrieved 4 chunks from 4 sources',
      'Few-shot: 0 example(s) retrieved',
      'Finance worker: resolved',
      'Escalation check: False',
      'QA PASS',
      'Finalized for Rajesh Kumar',
    ],
    "sources": ['Finance Policies > How to Claim', 'Finance Policies > Travel Policy', 'Finance Policies > Vendor Payments', 'Finance Policies > Eligible Expenses'],
    "answer": "[FINANCE] To claim for a laptop damaged during work travel, you should submit an expense claim under the 'hardware' category. Please ensure you attach the repair invoice or replacement receipt and submit the claim within 30 days o",
  },
  {
    "id": 'run-3', "question": "it's broken",
    "category": 'finance', "confidence": 8,
    "escalated": False, "fallback_used": False,
    # the trail as returned: this turn PRECEDED by the one before it.
    "audit": [
      'Supervisor: finance (conf: 8)',
      'RAG: retrieved 4 chunks from 4 sources',
      'Few-shot: 0 example(s) retrieved',
      'Finance worker: resolved',
      'Escalation check: False',
      'QA PASS',
      'Finalized for Rajesh Kumar',
      'Supervisor: finance (conf: 8)',
      'RAG: retrieved 4 chunks from 4 sources',
      'Few-shot: 0 example(s) retrieved',
      "Finance ReAct: 3 iteration(s), tools: submit_expense_claim({'amount': 0, 'category': 'hardware', 'description': 'Laptop damaged during work travel', 'receipt_count': 1}), submit_expense_claim({'amount': 1, 'category': 'hardware', 'description': 'Laptop damaged during work travel', 'receipt_count': 1})",
      'Finance worker: resolved',
      'Escalation check: False',
      'QA PASS',
      'Finalized for Rajesh Kumar',
    ],
    "sources": ['Finance Policies > Salary & Payroll', 'Finance Policies > Invoicing & Payments', 'Finance Policies > Expense Reimbursement', 'Finance Policies'],
    "answer": "[FINANCE] Your expense claim for the damaged laptop has been submitted successfully under the 'hardware' category. The Claim ID is EXP-2026-0007. Please remember to attach the repair invoice or replacement receipt and ensure the c",
  },
  {
    "id": 'run-4', "question": 'My salary slip shows the wrong VPN access for my desk booking',
    "category": 'tech', "confidence": 7,
    "escalated": False, "fallback_used": False,
    # the trail as returned: this turn PRECEDED by the one before it.
    "audit": [
      'Supervisor: finance (conf: 8)',
      'RAG: retrieved 4 chunks from 4 sources',
      'Few-shot: 0 example(s) retrieved',
      "Finance ReAct: 3 iteration(s), tools: submit_expense_claim({'amount': 0, 'category': 'hardware', 'description': 'Laptop damaged during work travel', 'receipt_count': 1}), submit_expense_claim({'amount': 1, 'category': 'hardware', 'description': 'Laptop damaged during work travel', 'receipt_count': 1})",
      'Finance worker: resolved',
      'Escalation check: False',
      'QA PASS',
      'Finalized for Rajesh Kumar',
      'Supervisor: tech (conf: 7)',
      'RAG: retrieved 4 chunks from 4 sources',
      'Few-shot: 0 example(s) retrieved',
      "Tech ReAct: 2 iteration(s), tools: create_ticket({'summary': 'Incorrect VPN access details on salary slip for desk booking', 'priority': 'P3', 'category': 'general', 'description': 'Employee reports that their salary slip displays incorrect VPN access information related to their desk booking.'})",
      'Tech worker: resolved',
      'Escalation check: False',
      'QA PASS',
      'Finalized for Rajesh Kumar',
    ],
    "sources": ['It Support > VPN Access', 'It Support > Troubleshooting', 'It Support > Setup', 'It Support > Security Policies'],
    "answer": '[TECH] I have created a ticket (TECH-1011) for this issue. Please note that salary slips are typically generated by the Finance/HR system, so this may require coordination between IT and HR to correct the underlying data. Sources:',
  },
  {
    "id": 'run-5', "question": 'I need 45 days of leave starting next Monday, approve it',
    "category": 'hr', "confidence": 9,
    "escalated": True, "fallback_used": False,
    # the trail as returned: this turn PRECEDED by the one before it.
    "audit": [
      'Supervisor: tech (conf: 7)',
      'RAG: retrieved 4 chunks from 4 sources',
      'Few-shot: 0 example(s) retrieved',
      "Tech ReAct: 2 iteration(s), tools: create_ticket({'summary': 'Incorrect VPN access details on salary slip for desk booking', 'priority': 'P3', 'category': 'general', 'description': 'Employee reports that their salary slip displays incorrect VPN access information related to their desk booking.'})",
      'Tech worker: resolved',
      'Escalation check: False',
      'QA PASS',
      'Finalized for Rajesh Kumar',
      'Supervisor: hr (conf: 9)',
      'RAG: retrieved 4 chunks from 4 sources',
      'Few-shot: 0 example(s) retrieved',
      "Hr ReAct: 3 iteration(s), tools: get_leave_balance_from_hr_system({}), get_leave_balance({}), apply_leave({'leave_type': 'casual', 'start_date': '2026-09-14', 'end_date': '2026-11-27', 'reason': 'Personal leave request for 45 days'})",
      'Hr worker: escalating',
      'Escalation check: True',
      'Manager called tool → Could not reach HR MCP server at http://mcp-leave.postgres.svc.cluster.local:8001/mcp: [Errno -2] Name or service not kn',
      'Manager called tool → Could not reach HR MCP server at http://mcp-leave.postgres.svc.cluster.local:8001/mcp: [Errno -2] Name or service not kn',
      'Manager resolved after 3 step(s)',
      'QA PASS',
      'Finalized for Rajesh Kumar',
    ],
    "sources": ['Hr Handbook > Casual Leave (CL)', 'Hr Handbook > Earned/Privilege Leave (EL)', 'Hr Handbook > Paternity Leave', 'Hr Handbook > Sick Leave (SL)'],
    "answer": "[HR] [Manager Approval] I understand you're requesting 45 days of leave starting next Monday (2026-09-14). However, I must decline this request for the following reasons:  1. **Exceeds maximum consecutive leave limit**: Per compan",
  },
  {
    "id": 'run-6', "question": 'What is the status of my open IT tickets?',
    "category": 'tech', "confidence": 9,
    "escalated": False, "fallback_used": False,
    # the trail as returned: this turn PRECEDED by the one before it.
    "audit": [
      'Supervisor: hr (conf: 9)',
      'RAG: retrieved 4 chunks from 4 sources',
      'Few-shot: 0 example(s) retrieved',
      "Hr ReAct: 3 iteration(s), tools: get_leave_balance_from_hr_system({}), get_leave_balance({}), apply_leave({'leave_type': 'casual', 'start_date': '2026-09-14', 'end_date': '2026-11-27', 'reason': 'Personal leave request for 45 days'})",
      'Hr worker: escalating',
      'Escalation check: True',
      'Manager called tool → Could not reach HR MCP server at http://mcp-leave.postgres.svc.cluster.local:8001/mcp: [Errno -2] Name or service not kn',
      'Manager called tool → Could not reach HR MCP server at http://mcp-leave.postgres.svc.cluster.local:8001/mcp: [Errno -2] Name or service not kn',
      'Manager resolved after 3 step(s)',
      'QA PASS',
      'Finalized for Rajesh Kumar',
      'Supervisor: tech (conf: 9)',
      'RAG: retrieved 4 chunks from 4 sources',
      'Few-shot: 0 example(s) retrieved',
      'Tech ReAct: 2 iteration(s), tools: list_my_tickets({})',
      'Tech worker: resolved',
      'Escalation check: False',
      'QA PASS',
      'Finalized for Rajesh Kumar',
    ],
    "sources": ['It Support > How to Raise a Ticket', 'It Support > Requesting New Software', 'It Support > Contact', 'It Support'],
    "answer": '[TECH] You have one open IT ticket: TECH-1004, which is for a slow-loading Jira dashboard. It is currently marked as Resolved. Sources: It Support > How to Raise a Ticket, It Support > Requesting New Software, It Support > Contact',
  },
]

print(len(RUNS), "runs loaded;", [len(r['audit']) for r in RUNS], "audit entries each")

## Section 1 &mdash; Find this run, then walk the ladder

### First: the audit trail is not one run

The trail the app returns grows with the conversation, because LangGraph's checkpointer keys on
the user and the `audit` field is an appending list. Across the six captures it went from 102
entries to 144. So before you can read a run you have to find where it starts &mdash; at the last
`Supervisor:` line, which is the first thing every request does.

That is not a detail. Read the whole array and you will diagnose a failure that happened twenty
minutes ago, in a different request, for a different question.

### Then: check the cheap things first

Eight steps, in the order the request goes through them. The point of a fixed order is that each
rung is cheaper to check than the one below, and a failure at any rung makes everything under it
unreliable &mdash; so the **first** rung that fails is the one to fix.

| Rung | What you are asking | Where you see it |
|---|---|---|
| `route` | did the supervisor pick the right department? | `Supervisor: <category> (conf: N)` |
| `clarify` | was it confident enough to answer at all? | confidence, and whether it asked a question instead |
| `retrieve` | did RAG return anything, and is it relevant? | `RAG: retrieved N chunks`, plus the source list |
| `tool_pick` | were the right tools chosen &mdash; any at all? | `<Worker> ReAct: ... tools: ...` |
| `tool_run` | did those tools actually succeed? | an error string in the trail |
| `converge` | did the worker finish, or run out of turns? | `N iteration(s)` against a limit of 3 |
| `write` | did it change something, and was it asked to? | a write tool in the tool list |
| `qa` | did the gate catch anything? | `QA PASS` / a fallback |

In [ ]:
# ---------------------------------------------------- reading ONE run, given
# The trail accumulates across a conversation, so a run starts at the last
# "Supervisor:" line. Everything before that belongs to an earlier request.
def this_run(run: dict) -> list:
    a = run["audit"]
    start = max((i for i, l in enumerate(a) if l.startswith("Supervisor:")), default=0)
    return a[start:]

def tools_called(run: dict) -> list:
    """Every tool name in this run's ReAct lines, in order."""
    names = []
    for line in this_run(run):
        if "tools:" in line:
            for part in line.split("tools:", 1)[1].split("),"):
                name = part.split("(")[0].strip()
                if name:
                    names.append(name)
    return names

def show(run: dict) -> None:
    print(f"{run['id']}  {run['question']!r}")
    print(f"  category={run['category']} confidence={run['confidence']} "
          f"escalated={run['escalated']} fallback={run['fallback_used']}")
    for line in this_run(run):
        print("   ", line)
    print("  answer:", run["answer"][:150])

show(RUNS[0])       # the healthy one -- your baseline

In [ ]:
# -------------------------------------------------- the eight rungs, given
# One predicate per rung. Each returns True when that step looks HEALTHY.
# These read only the audit trail -- no model, no cluster, no network.
WRITE_TOOLS = {"apply_leave", "approve_leave_via_mcp", "submit_expense_claim",
               "approve_expense_claim", "create_ticket", "book_meeting_room",
               "send_email", "install_skill", "write_local_file"}

RUNGS = ["route", "clarify", "retrieve", "tool_pick", "tool_run", "converge", "write", "qa"]

def rung_route(run):
    if run["id"] in UNANSWERABLE:        # no department is right; see rung_clarify
        return True
    return run["category"] in EXPECTED_CATEGORY.get(run["id"], {run["category"]})

def rung_clarify(run):
    """Is the confidence plausible GIVEN the question?

    The app clarifies below 5. That gate cannot help when the score itself is
    wrong -- so judge the score against how much there was to go on.
    """
    words = [w for w in _re.findall(r"[a-zA-Z]+", run["question"]) if len(w) > 2]
    if len(words) <= 2:                  # almost nothing to classify on
        return run["confidence"] < 5     # it should have asked, not answered
    return run["confidence"] >= 5
def rung_retrieve(run):  return any("RAG: retrieved" in l and "0 chunks" not in l
                                    for l in this_run(run))
def rung_tool_pick(run): return bool(tools_called(run)) or run["id"] in NO_TOOL_NEEDED
def rung_tool_run(run):  return not any("Could not reach" in l or "error" in l.lower()
                                        for l in this_run(run))
def rung_converge(run):  return not any("3 iteration(s)" in l for l in this_run(run))
def rung_write(run):     return not (set(tools_called(run)) & WRITE_TOOLS) or run["id"] in WRITE_ASKED_FOR
def rung_qa(run):        return not run["fallback_used"]

PREDICATE = {"route": rung_route, "clarify": rung_clarify, "retrieve": rung_retrieve,
             "tool_pick": rung_tool_pick, "tool_run": rung_tool_run,
             "converge": rung_converge, "write": rung_write, "qa": rung_qa}

# What a correct system would have done. Hand-labelled, which is what a ground
# truth is -- somebody decided, and you can disagree with them.
EXPECTED_CATEGORY = {"run-1": {"hr"}, "run-2": {"finance"}, "run-3": {"hr", "tech", "facilities"},
                     "run-4": {"finance", "hr"}, "run-5": {"hr"}, "run-6": {"tech"}}
NO_TOOL_NEEDED    = {"run-2"}                 # a policy question RAG can answer
UNANSWERABLE      = {"run-3"}                 # "it's broken" -- no department is the right one
WRITE_ASKED_FOR   = {"run-5"}                 # "approve it" does ask for a write

def first_failing_rung(run: dict) -> str | None:
    """The first rung, in order, whose step does not look healthy."""
    for name in RUNG_ORDER():
        if not PREDICATE[name](run):
            return name
    return None

In [ ]:
# ------------------------------------------------------------ your decisions
def RUNG_ORDER() -> list:
    """The order to check the rungs in.

    `RUNGS` above is already in the order a request passes through the steps.
    Checking in that order means the first failure you find is the earliest one,
    and everything below it is downstream of a broken input.
    """
    return BLANK                      # RUNGS  |  sorted(RUNGS)  |  RUNGS[::-1]


def stops_the_two_word_message() -> str:
    """run-3 is the message "it's broken". It ended up submitting an expense claim.

    Two rungs could each have stopped that on their own. Which one is EARLIER,
    and so the one to fix first? Return a rung name from RUNGS.
    """
    return BLANK                      # "clarify" | "write" | "qa"


def qa_pass_means_correct() -> bool:
    """All six runs printed QA PASS, and four of them are wrong.

    So: does QA PASS tell you the run was correct?
    """
    return BLANK                      # True | False


def worst_of_the_six() -> str:
    """One run did something that cannot be undone by re-asking the question.

    Escalating a bad answer costs a conversation. Escalating THIS costs a
    correction in a system of record. Return its id.
    """
    return BLANK                      # "run-3" | "run-4" | "run-5" | "run-6"

In [ ]:
# --- Self-check: the ladder over six real runs  (audit trails only -- no model)
check("the ladder is walked in request order",
      lambda: RUNG_ORDER() == RUNGS,
      "sorted() is alphabetical, and reversed() finds the LAST failure, not the first")

check("run-2 is the clean one, all eight rungs",
      lambda: first_failing_rung(RUNS[1]) is None,
      "a policy question answered from retrieval, no tools, nothing written")

check("run-1 -- the one that looks fine -- fails at converge",
      lambda: first_failing_rung(RUNS[0]) == "converge",
      "3 of 3 ReAct iterations for a balance lookup, and it called two leave tools")

check("run-3 -- 'it's broken' -- fails at clarify, not later",
      lambda: first_failing_rung(RUNS[2]) == "clarify",
      "confidence 8 on a two-word message. Everything after that is downstream")

check("...and the earlier of its two possible stops is the one to fix",
      lambda: stops_the_two_word_message() == "clarify")

check("run-5 -- the 45-day request -- fails at tool_run",
      lambda: first_failing_rung(RUNS[4]) == "tool_run",
      "'Could not reach HR MCP server', twice, and the manager still answered")

check("run-3 made a write nobody asked for",
      lambda: bool(set(tools_called(RUNS[2])) & WRITE_TOOLS)
              and RUNS[2]["id"] not in WRITE_ASKED_FOR,
      "submit_expense_claim, off a two-word message, with an invented description")

check("run-6 passes every rung and is still wrong",
      lambda: first_failing_rung(RUNS[5]) is None
              and "open" in RUNS[5]["answer"].lower()
              and "resolved" in RUNS[5]["answer"].lower(),
      "one OPEN ticket that is currently RESOLVED. No rung here reads the answer")

check("QA PASS is not a statement about correctness",
      lambda: qa_pass_means_correct() is False
              and all("QA PASS" in " ".join(this_run(r)) for r in RUNS))

check("the run to escalate is the one that wrote to a system of record",
      lambda: worst_of_the_six() == "run-3",
      "a wrong answer can be re-asked; EXP-2026-0007 has to be withdrawn")

check("every run is diagnosed from its own turn, not the whole conversation",
      lambda: all(len(this_run(r)) < len(r["audit"]) for r in RUNS)
              and all(this_run(r)[0].startswith("Supervisor:") for r in RUNS),
      "each trail here carries the previous turn too -- cut at the LAST Supervisor line")

score()

## Section 2 &mdash; What the failure cost, and which one to fix first

Two runs can fail at the same rung and be worth very different amounts of your attention.

A failure at `route` throws away the whole request &mdash; every call after it was spent on the
wrong department. A failure at `qa` throws away almost nothing, because everything before it was
useful work. So *where* a run failed tells you how much it wasted, and you can read that straight
off the trail without any extra instrumentation.

Then one thing outranks all of that: whether the run **changed something**. A wrong answer costs
a conversation. A wrong write costs a correction in a system of record, and somebody else's
afternoon.

In [ ]:
# ------------------------------------------------- cost and priority, given
def steps_before(run: dict, rung: str) -> int:
    """How many rungs the request got through before the one that failed."""
    return RUNGS.index(rung) if rung in RUNGS else len(RUNGS)

def wasted(run: dict) -> int:
    """Model calls spent before the failure. One per ReAct iteration, plus the
    supervisor, plus the worker's final answer."""
    iters = sum(int(l.split(" iteration(s)")[0].split()[-1])
                for l in this_run(run) if "iteration(s)" in l)
    return 1 + iters + 1

def report():
    print(f"{'run':7} {'failed at':11} {'got through':11} {'calls':5}  wrote?")
    for r in RUNS:
        rung = first_failing_rung(r)
        wrote = bool(set(tools_called(r)) & WRITE_TOOLS)
        print(f"{r['id']:7} {str(rung or 'healthy'):11} "
              f"{steps_before(r, rung or ''):>11} {wasted(r):>5}  "
              f"{'YES' if wrote else '-'}")

guard(report)

## Section 3 &mdash; Break your own deployment, then locate it

Reading somebody else's failures is practice. Causing one and finding it is the skill.

These cells are marked **Run it for real** and need the app deployed in your namespace
(Module 9's lab, or `bash scripts/deploy-spark.sh` from your clone). They self-skip with
instructions if it is not there, so the rest of the lab still scores.

You will break **one** step, ask a question that depends on it, and walk the ladder on the
result. The failure you are about to cause is the same one as `run-5`: point the app at an HR
system that does not answer.

In [ ]:
# --- Run it for real: break one step, then find it ---------------------------
import subprocess

def kubectl(*args):
    return subprocess.run(("kubectl", "-n", APP_NS or "none") + args,
                          capture_output=True, text=True)

def deployed() -> bool:
    if not APP_NS:
        print("APP_NAMESPACE is unset, so there is no deployment to break.")
        print("Deploy it first (Module 9), then re-run this cell.")
        return False
    got = kubectl("get", "deploy", "frontdeskai", "-o", "name")
    if got.returncode != 0:
        print("frontdeskai is not deployed in", APP_NS, "-- deploy it first.")
        return False
    return True

def break_the_hr_tool():
    """Point MCP_LEAVE_URL at a host that does not resolve, and roll it out."""
    if not deployed():
        return
    out = kubectl("patch", "configmap", "frontdeskai-config", "--type", "merge",
                  "-p", json.dumps({"data": {"MCP_LEAVE_URL":
                                    "http://no-such-hr-system.invalid:8001/mcp"}}))
    print(out.stdout.strip() or out.stderr.strip())
    kubectl("rollout", "restart", "deploy/frontdeskai")
    print(kubectl("rollout", "status", "deploy/frontdeskai", "--timeout=300s").stdout.strip())
    print("\nNow ask it 'what is my leave balance?' -- in the browser, or the cell below.")

guard(break_the_hr_tool)

In [ ]:
# --- Run it for real: ask, capture the trail, and walk your own ladder -------
ASK_PY = (
    "import json,urllib.request,urllib.parse,http.cookiejar;"
    "j=http.cookiejar.CookieJar();"
    "o=urllib.request.build_opener(urllib.request.HTTPCookieProcessor(j));"
    "p=lambda u,d: o.open('http://127.0.0.1:8000'+u,"
    " data=urllib.parse.urlencode(d).encode(), timeout=600);"
    "p('/login', {'email':'rajesh.kumar@unigps.in','password':'brainupgrade'});"
    "r=json.loads(p('/chat/send', {'message':'What is my current leave balance?'}).read());"
    "print(json.dumps({'category':r['category'],'confidence':r['confidence'],"
    "'escalated':r['escalated'],'fallback_used':r['fallback_used'],"
    "'audit':r['audit'],'sources':r.get('sources',[]),'answer':r['response'][:230]}))"
)

def diagnose_live():
    if not deployed():
        return
    out = kubectl("exec", "deploy/frontdeskai", "--", "python3", "-c", ASK_PY)
    if out.returncode != 0:
        print("the request did not complete:", (out.stderr or out.stdout).strip()[:300]); return
    mine = json.loads(out.stdout.strip().splitlines()[-1])
    mine["id"] = "mine"
    EXPECTED_CATEGORY["mine"] = {"hr"}
    show(mine)
    print()
    print("first failing rung ->", first_failing_rung(mine))
    print()
    print("Read the answer again. Does it admit that anything failed?")

guard(diagnose_live)

In [ ]:
# --- Run it for real: fix exactly that step, and prove the rung clears -------
def put_it_back():
    if not deployed():
        return
    out = kubectl("patch", "configmap", "frontdeskai-config", "--type", "merge",
                  "-p", json.dumps({"data": {"MCP_LEAVE_URL":
                                    "http://mcp-leave.postgres.svc.cluster.local:8001/mcp"}}))
    print(out.stdout.strip() or out.stderr.strip())
    kubectl("rollout", "restart", "deploy/frontdeskai")
    print(kubectl("rollout", "status", "deploy/frontdeskai", "--timeout=300s").stdout.strip())
    print("\nRe-run the cell above. tool_run should clear -- and note what did NOT change:")
    print("the category, the retrieval, the QA verdict. You fixed one rung, not the system.")

guard(put_it_back)

In [ ]:
score()

## Your turn

1. **Disagree with the ground truth.** `EXPECTED_CATEGORY` says run-4 &mdash; *&ldquo;my salary
   slip shows the wrong VPN access for my desk booking&rdquo;* &mdash; should have gone to finance
   or hr, and the app chose tech with confidence 7. Make the case for tech. If you win the
   argument, the label is wrong, not the app, and you have just found the most common defect in
   an eval set.
2. **Add the rung this ladder does not have.** Nothing here checks whether the retrieved sources
   are *relevant* to the question, only that some arrived. Run-3 retrieved four finance chunks
   for a two-word message and `retrieve` passed. Write that rung, and say honestly what it costs
   to evaluate.
3. **Break a different step.** Upload a policy document that contradicts the handbook, then ask a
   question it covers. Every tool succeeds, every rung passes, and the answer is wrong &mdash;
   which rung would have to exist to catch it, and can it be checked without a model?
4. **Take run-3 apart properly.** It invented *&ldquo;Laptop damaged during work travel&rdquo;*
   from the previous turn in the conversation, then submitted it twice with `amount: 0` and
   `amount: 1`. Three separate defects are visible in that one trail. Name them, and say which
   rung each belongs to.

> **What you take from Module 7:** a failure has a location, and finding it is cheaper than
> arguing about it. The audit trail is not one run until you cut it. The first rung that fails is
> the one to fix, because everything under it was working from a broken input. And `QA PASS` is a
> statement about a gate having run, not about the answer being right.